In [ ]:
from huggingface_hub import login
token=''
login(token)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import inspect

model = AutoModel.from_pretrained(
    "facebook/dinov3-vitb16-pretrain-lvd1689m"
)

print("=== model loaded ===")
print(model)


In [ ]:
from src.make_gate import inject_gating, freeze_except_gate

In [ ]:
model, hooks = inject_gating(model, gate_type="elementwise")

In [ ]:
from src.model import dinosplus_classfier
vit=dinosplus_classfier(model,200)


In [ ]:
print(vit)

In [ ]:
for i in vit.parameters():
    i.requires_grad=True

In [ ]:
from datasets import load_dataset
from src.data import setdata
from torch.utils.data import DataLoader
data=load_dataset('zh-plus/tiny-imagenet')
train=setdata(data['train'])
test=setdata(data['valid'])
train_loader=DataLoader(train,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)
test_loader=DataLoader(test,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch
optimizer=optim.Adam(vit.parameters(),lr=0.0001)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler()

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
device= 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import torch

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [ ]:
from src.training import train 
train(vit,train_loader,test_loader,50,criterion,scaler,device,optimizer,'full fine tuning vit dino v3 B gated','/home/hyuksu/deep-learning-study/outputs/best dino v3 B gated.pth')